# Documentation for which methods are applied and why

In [1]:
import pandas as pd
from pathlib import Path
from IPython.display import display

methods_table = pd.DataFrame([
    {
        "Step": "0. Data inspection",
        "What happens": "Check columns, coverage and distributions; choose one primary genre per song",
        "Course method(s)": "– (data cleaning)",
        "Source in course": "–",
        "Why this step and method": "Every later choice depends on what the data contains. Missing instruments or messy genres change what we can conclude.",
        "Notes": "Report coverage (e.g. 'X% of songs have instrument data'). Note the genre mapping choice as a limitation.",
    },
    {
        "Step": "1a. G_lyrics",
        "What happens": "TF-IDF on tokenized lyrics → cosine similarity → song–song graph",
        "Course method(s)": "TF-IDF, cosine similarity",
        "Source in course": "Text-analysis part; cosine as a projection weight in book ch. 26",
        "Why this step and method": "TF-IDF picks out the words characteristic of each song, not the ones every song uses. Cosine compares songs regardless of lyric length.",
        "Notes": "min_df=2 removes words that can't link two songs; max_df=0.5 removes near-stop words. IDF follows the same logic as hyperbolic weighting: common features count less.",
    },
    {
        "Step": "1b. G_instruments",
        "What happens": "Song–instrument bipartite → hyperbolic projection onto songs",
        "Course method(s)": "Bipartite networks, hyperbolic projection",
        "Source in course": "Book ch. 26",
        "Why this step and method": "Songs and instruments are naturally bipartite. Hyperbolic weighting down-weights very common instruments (electric guitar), so rare shared instruments count for more.",
        "Notes": "Only songs with instrument data take part. Plain co-occurrence counting would turn the network into a hairball.",
    },
    {
        "Step": "1c. G_combined",
        "What happens": "Bipartite song–(instruments + country + top TF-IDF words) → hyperbolic projection",
        "Course method(s)": "Bipartite projection",
        "Source in course": "Book ch. 26",
        "Why this step and method": "Matches the research question: cluster on all three variables at once. Hyperbolic weighting automatically weakens the influence of big countries (UK/US).",
        "Notes": "No country-only graph: it would just be one isolated clique per country. 'Top-10 TF-IDF words' is an operationalisation of 'lyrical themes' and must be justified.",
    },
    {
        "Step": "2. Backboning",
        "What happens": "Remove edges that aren't statistically significant",
        "Course method(s)": "Disparity filter",
        "Source in course": "Book ch. 27 (27.5)",
        "Why this step and method": "Projections and similarity graphs are very dense. A fixed threshold fails because edge weights are broadly distributed; the disparity filter tests each edge against what's expected for that node.",
        "Notes": "alpha is a choice, so test 0.01/0.05/0.1 (step 6C). Report how many songs/edges survive and the size of the largest component.",
    },
    {
        "Step": "3. Descriptive statistics",
        "What happens": "Degree distribution, average degree, clustering coefficient, components",
        "Course method(s)": "CCDF on log-log axes, clustering coefficient, components",
        "Source in course": "First half of the course (book ch. 9 on degree)",
        "Why this step and method": "Shows what kind of network is being clustered, and lets the reader judge the graphs.",
        "Notes": "Use the CCDF rather than the raw distribution for heavy tails. Don't claim a power law without proper fitting (powerlaw package / MLE).",
    },
    {
        "Step": "4. Community detection",
        "What happens": "Find clusters of songs",
        "Course method(s)": "Louvain (modularity maximisation)",
        "Source in course": "Book ch. 36–37 (37.1)",
        "Why this step and method": "The course's standard community-detection method: fast, works on weighted graphs, and gives a modularity M that can be compared against a null model.",
        "Notes": "Resolution limit: modularity has a preferred community size (≈ √|E| communities), so small genres can get merged. Louvain is random, hence step 6C.",
    },
    {
        "Step": "5a. AMI / ARI",
        "What happens": "Compare the community partition with genre and with country",
        "Course method(s)": "Adjusted mutual information, Adjusted Rand Index",
        "Source in course": "Book ch. 36 (mutual information from 3.5)",
        "Why this step and method": "Measures how much knowing a song's cluster tells us about its genre. The adjusted versions are 0 at chance level.",
        "Notes": "Don't use plain NMI: random vectors still get NMI ≈ 0.09. Compute on the songs common to all graphs for a fair comparison.",
    },
    {
        "Step": "5b. Confusion heatmap",
        "What happens": "Communities × genres, row-normalised",
        "Course method(s)": "Visualising partition vs metadata",
        "Source in course": "Book ch. 36 (communities vs node metadata)",
        "Why this step and method": "AMI is one number. The heatmap shows which clusters match which genres.",
        "Notes": "Most useful figure for the discussion. Do the same for country.",
    },
    {
        "Step": "5c. Assortativity",
        "What happens": "Do songs link to songs of the same genre/country?",
        "Course method(s)": "Attribute assortativity",
        "Source in course": "Book ch. 30 (homophily)",
        "Why this step and method": "Answers the association question at edge level, without depending on Louvain.",
        "Notes": "Ranges from −1 to 1; 0 = no more same-label links than chance. A good complement to AMI.",
    },
    {
        "Step": "5d. Genre modularity",
        "What happens": "Compute M using genre as the partition, compare with Louvain's M",
        "Course method(s)": "Modularity",
        "Source in course": "Book ch. 36",
        "Why this step and method": "Direct test of how well genre explains the network structure, relative to the best partition found.",
        "Notes": "If M_genre ≈ M_Louvain, genre structures the network almost as well as the best partition Louvain found. Decide how songs without a genre are handled.",
    },
    {
        "Step": "6A. Configuration-model null",
        "What happens": "~100 degree-preserving randomisations → Louvain → distribution of M",
        "Course method(s)": "Configuration model, double-edge swap",
        "Source in course": "Book ch. 18.1 + ch. 36",
        "Why this step and method": "Tests whether the community structure is genuine or would arise just from the degrees. Random fluctuations alone can raise modularity.",
        "Notes": "Report z-score and empirical p-value. For weighted graphs, shuffle weights onto the rewired edges (a simplification, so mention it).",
    },
    {
        "Step": "6B. Permutation test",
        "What happens": "Shuffle genre labels 1000× → distribution of AMI",
        "Course method(s)": "Permutation-based null model",
        "Source in course": "Same null-model idea as ch. 36",
        "Why this step and method": "Gives a p-value for whether the genre agreement exceeds chance, as a supplement to AMI's built-in correction.",
        "Notes": "Do the same for country. Cheap to run, since no network work is involved.",
    },
    {
        "Step": "6C. Stability",
        "What happens": "Louvain with 20 seeds + pipeline with several alpha values",
        "Course method(s)": "Partition comparison with AMI",
        "Source in course": "Book ch. 36",
        "Why this step and method": "Clusters that change from run to run, or with small parameter changes, shouldn't be interpreted.",
        "Notes": "Mean pairwise AMI above ~0.8 = robust. Report how M, number of communities and AMI(genre) vary with alpha.",
    },
    {
        "Step": "6D. Content check",
        "What happens": "TF-IDF per community → word clouds; sentiment per community/genre",
        "Course method(s)": "TF-IDF, word clouds, sentiment (VADER/LabMT)",
        "Source in course": "Text-analysis part",
        "Why this step and method": "Qualitative check: do the clusters make thematic sense? This is what makes them 'meaningful', not only statistically significant.",
        "Notes": "Only communities with ≥10 songs. VADER is built for raw text, so joined tokens are an approximation; LabMT works directly on words.",
    },
    {
        "Step": "7. Associations",
        "What happens": "Instrument × country and instrument × theme",
        "Course method(s)": "Contingency tables, lift, chi-square",
        "Source in course": "Basic statistics (supports the course work; not a network method)",
        "Why this step and method": "Answers sub-question 2 directly: which instruments go with which countries and themes.",
        "Notes": "Lift > 1 = more often together than expected. Rare combinations give unstable lift, so require a minimum count.",
    },
    {
        "Step": "8. Summary table",
        "What happens": "One row per graph with all key numbers",
        "Course method(s)": "– (collects steps 4–6)",
        "Source in course": "–",
        "Why this step and method": "Answers 'to what extent': which variables give genuine clusters (high z, stable) and which match genre (AMI).",
        "Notes": "Main results table in the PNAS report. A low AMI is a result too: metadata and structural communities often diverge.",
    },
])
methods_table = methods_table.drop(columns=["Source in course"])

# Readable display in the notebook: wrapped text, left-aligned, no index
styled = (
    methods_table.style
    .hide(axis="index")
    .set_properties(**{"text-align": "left", "white-space": "pre-wrap", "vertical-align": "top"})
    .set_table_styles([
        {"selector": "th", "props": [("text-align", "left"), ("vertical-align", "top")]},
        {"selector": "td", "props": [("max-width", "260px")]},
    ])
)
display(styled)

# Save a copy for the group / the report
Path("results").mkdir(exist_ok=True)
methods_table.to_csv("results/methods_table.csv", index=False)

Step,What happens,Course method(s),Source in course,Why this step and method,Notes
0. Data inspection,"Check columns, coverage and distributions; choose one primary genre per song",– (data cleaning),–,Every later choice depends on what the data contains. Missing instruments or messy genres change what we can conclude.,Report coverage (e.g. 'X% of songs have instrument data'). Note the genre mapping choice as a limitation.
1a. G_lyrics,TF-IDF on tokenized lyrics → cosine similarity → song–song graph,"TF-IDF, cosine similarity",Text-analysis part; cosine as a projection weight in book ch. 26,"TF-IDF picks out the words characteristic of each song, not the ones every song uses. Cosine compares songs regardless of lyric length.",min_df=2 removes words that can't link two songs; max_df=0.5 removes near-stop words. IDF follows the same logic as hyperbolic weighting: common features count less.
1b. G_instruments,Song–instrument bipartite → hyperbolic projection onto songs,"Bipartite networks, hyperbolic projection",Book ch. 26,"Songs and instruments are naturally bipartite. Hyperbolic weighting down-weights very common instruments (electric guitar), so rare shared instruments count for more.",Only songs with instrument data take part. Plain co-occurrence counting would turn the network into a hairball.
1c. G_combined,Bipartite song–(instruments + country + top TF-IDF words) → hyperbolic projection,Bipartite projection,Book ch. 26,Matches the research question: cluster on all three variables at once. Hyperbolic weighting automatically weakens the influence of big countries (UK/US).,No country-only graph: it would just be one isolated clique per country. 'Top-10 TF-IDF words' is an operationalisation of 'lyrical themes' and must be justified.
2. Backboning,Remove edges that aren't statistically significant,Disparity filter,Book ch. 27 (27.5),Projections and similarity graphs are very dense. A fixed threshold fails because edge weights are broadly distributed; the disparity filter tests each edge against what's expected for that node.,"alpha is a choice, so test 0.01/0.05/0.1 (step 6C). Report how many songs/edges survive and the size of the largest component."
3. Descriptive statistics,"Degree distribution, average degree, clustering coefficient, components","CCDF on log-log axes, clustering coefficient, components",First half of the course (book ch. 9 on degree),"Shows what kind of network is being clustered, and lets the reader judge the graphs.",Use the CCDF rather than the raw distribution for heavy tails. Don't claim a power law without proper fitting (powerlaw package / MLE).
4. Community detection,Find clusters of songs,Louvain (modularity maximisation),Book ch. 36–37 (37.1),"The course's standard community-detection method: fast, works on weighted graphs, and gives a modularity M that can be compared against a null model.","Resolution limit: modularity has a preferred community size (≈ √|E| communities), so small genres can get merged. Louvain is random, hence step 6C."
5a. AMI / ARI,Compare the community partition with genre and with country,"Adjusted mutual information, Adjusted Rand Index",Book ch. 36 (mutual information from 3.5),Measures how much knowing a song's cluster tells us about its genre. The adjusted versions are 0 at chance level.,Don't use plain NMI: random vectors still get NMI ≈ 0.09. Compute on the songs common to all graphs for a fair comparison.
5b. Confusion heatmap,"Communities × genres, row-normalised",Visualising partition vs metadata,Book ch. 36 (communities vs node metadata),AMI is one number. The heatmap shows which clusters match which genres.,Most useful figure for the discussion. Do the same for country.
5c. Assortativity,Do songs link to songs of the same genre/country?,Attribute assortativity,Book ch. 30 (homophily),"Answers the association question at edge level, without depending on Louvain.",Ranges from −1 to 1; 0 = no more same-label links than chance. A good complement to AMI.
